Task 1: Conceptual Understanding

1. What is the difference between "Love" and "love" in NLP?
In NLP, raw text is strictly case-sensitive. To a computer, "Love" (with a capital 'L') and "love" (with a lowercase 'l') have different ASCII/Unicode values and are treated as two entirely different tokens. This creates unnecessary duplication in the vocabulary. Converting text to a standard case (usually lowercase) normalizes the data so the model recognizes both variations as the same semantic word.

2. What happens if stopwords are not removed?
If stopwords (common words like "is", "the", "and") are not removed, they will dominate the frequency distribution of the dataset. This increases the feature space and memory consumption significantly. Furthermore, in models like TF-IDF or simple Bag of Words, these highly frequent but semantically meaningless words can overshadow the important, context-rich words, potentially confusing the machine learning model.

3. Provide two real-world scenarios where removing stopwords can be harmful.


Sentiment Analysis: Words like "not" or "no" are technically stopwords, but they completely invert the meaning of a sentence. If removed, "I am not happy" becomes "I am happy," entirely ruining the model's prediction.


Machine Translation / Text Generation: When translating text from one language to another, or generating conversational AI responses, stopwords are crucial for grammatical structure and syntax. Removing them results in broken, robotic sentences.

4. Explain the difference between stemming and lemmatization.


Stemming: A crude, rule-based heuristic process that chops off the suffixes of words to find their root. It is fast but often results in non-dictionary words (e.g., "studying" might become "studi").


Lemmatization: A more advanced, dictionary-based process that uses morphological analysis to return the base, grammatically correct dictionary form of a word, known as the "lemma"  (e.g., "better" becomes "good"). It is slower but contextually accurate.

In [2]:
import re
from collections import Counter

# ==========================================
# Task 2: Build Advanced Preprocessing Function
# ==========================================
def preprocess_text(text):
    """
    Cleans and preprocesses raw text by removing noise, numbers, short tokens, 
    URLs, and normalizing repeated characters.
    """
    # Task 7 Integration: Error Handling for empty strings or invalid types
    if not text or not isinstance(text, str):
        return [], ""

    # 6. Remove URLs and email-like patterns
    text = re.sub(r'http\S+|www\.\S+|\S+@\S+', '', text)
    
    # 1. Remove numbers & Task 7 (Remove Emojis/Special Chars)
    # Replacing everything that is not an alphabet letter or whitespace with a space
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # 4. Convert text to lowercase
    text = text.lower()
    
    # 3. Handle repeated characters (e.g., "soooo" -> "so", "looooved" -> "loved")
    # Reduces 3 or more consecutive identical characters to just 1
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    
    # Tokenize the string
    tokens = text.split()
    
    # 5. Remove very short tokens (length <= 2), with exceptions
    exceptions = {"no", "not", "ok"}
    cleaned_tokens = [word for word in tokens if len(word) > 2 or word in exceptions]
    
    # 2. Remove extra spaces (handled cleanly by joining our filtered tokens)
    cleaned_sentence = " ".join(cleaned_tokens)
    
    return cleaned_tokens, cleaned_sentence

# ==========================================
# Task 3: Stress Testing
# ==========================================
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

print("--- TASK 3: STRESS TESTING ---")
for text in sample_inputs:
    tokens, cleaned_sent = preprocess_text(text)
    print(f"Original Text    : {text}")
    print(f"Cleaned Tokens   : {tokens}")
    print(f"Cleaned Sentence : {cleaned_sent}\n")


# ==========================================
# Task 4: Token Analytics
# ==========================================
print("--- TASK 4: TOKEN ANALYTICS ---")
analytics_data = []
all_global_tokens = [] # Used for Task 5

for text in sample_inputs:
    tokens, cleaned_sent = preprocess_text(text)
    all_global_tokens.extend(tokens)
    
    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    avg_length = sum(len(t) for t in tokens) / total_tokens if total_tokens > 0 else 0
    
    # To answer the analysis questions, we track noise dropped and tokens kept
    original_char_length = len(text)
    cleaned_char_length = len(cleaned_sent)
    noise_removed = original_char_length - cleaned_char_length
    
    analytics_data.append({
        "sentence": text,
        "total_tokens": total_tokens,
        "unique_tokens": unique_tokens,
        "avg_length": round(avg_length, 2),
        "noise_removed": noise_removed
    })

for data in analytics_data:
    print(f"Text: '{data['sentence']}'")
    print(f"  Total Tokens: {data['total_tokens']} | Unique: {data['unique_tokens']} | Avg Len: {data['avg_length']}")

# Answers to Analysis Questions (Using Print function as requested)
most_noise_sentence = max(analytics_data, key=lambda x: x['noise_removed'])
most_meaningful_sentence = max(analytics_data, key=lambda x: x['total_tokens'])

print("\n--- TASK 4: ANALYSIS QUESTIONS ---")
print(f"Q: Which sentence had the most noise? \nA: '{most_noise_sentence['sentence']}' (Removed {most_noise_sentence['noise_removed']} characters of noise/URLs/numbers)")
print(f"Q: Which sentence retained the most meaningful tokens after cleaning? \nA: '{most_meaningful_sentence['sentence']}' (Retained {most_meaningful_sentence['total_tokens']} tokens)")


# ==========================================
# Task 5: Frequency Analysis
# ==========================================
print("\n--- TASK 5: FREQUENCY ANALYSIS ---")
token_counts = Counter(all_global_tokens)

top_10_most = token_counts.most_common(10)
# To get the least frequent, we look at the end of the most_common list
top_5_least = token_counts.most_common()[:-6:-1] 

print("Top 10 most frequent words:")
for word, count in top_10_most:
    print(f"  {word}: {count}")

print("\nTop 5 least frequent words:")
for word, count in top_5_least:
    print(f"  {word}: {count}")


# ==========================================
# Task 6: Build Full Pipeline
# ==========================================
def full_pipeline(text_list):
    """
    Processes a list of text strings and returns a dictionary 
    with lists of all cleaned tokens and clean sentences.
    """
    all_tokens_pipeline = []
    all_clean_sentences = []
    
    for text in text_list:
        tokens, cleaned_sent = preprocess_text(text)
        all_tokens_pipeline.append(tokens)
        all_clean_sentences.append(cleaned_sent)
        
    return {
        "tokens": all_tokens_pipeline,
        "clean_sentences": all_clean_sentences
    }

print("\n--- TASK 6: FULL PIPELINE EXECUTION ---")
pipeline_result = full_pipeline(sample_inputs[:2]) # Testing with just the first 2 for brevity
print("Pipeline Output:")
print(f"tokens: {pipeline_result['tokens']}")
print(f"clean_sentences: {pipeline_result['clean_sentences']}")


# ==========================================
# Task 7: Error Handling 
# ==========================================
print("\n--- TASK 7: ERROR HANDLING TESTS ---")
error_test_cases = [
    "",                   # Empty string
    "🔥🔥🔥 😊😊",           # Only emojis
    "12345 67890",        # Only numbers
    None                  # Invalid data type
]

for test in error_test_cases:
    tokens, cleaned_sent = preprocess_text(test)
    print(f"Input: {repr(test)}")
    print(f"  Output Tokens: {tokens} | Output Sentence: '{cleaned_sent}'")

--- TASK 3: STRESS TESTING ---
Original Text    : Get 100% FREE access now!!!
Cleaned Tokens   : ['get', 'free', 'access', 'now']
Cleaned Sentence : get free access now

Original Text    : I absolutely looooved this product
Cleaned Tokens   : ['absolutely', 'loved', 'this', 'product']
Cleaned Sentence : absolutely loved this product

Original Text    : Worst service ever... 0/10
Cleaned Tokens   : ['worst', 'service', 'ever']
Cleaned Sentence : worst service ever

Original Text    : Call me at 9876543210
Cleaned Tokens   : ['call']
Cleaned Sentence : call

Original Text    : This is THE best course!!!
Cleaned Tokens   : ['this', 'the', 'best', 'course']
Cleaned Sentence : this the best course

Original Text    : Visit https://openai.com now!
Cleaned Tokens   : ['visit', 'now']
Cleaned Sentence : visit now

Original Text    : Nooooo this is baaad!!!
Cleaned Tokens   : ['no', 'this', 'bad']
Cleaned Sentence : no this bad

Original Text    : OK OK OK I got it
Cleaned Tokens   : ['ok', 'ok